# 06 Silver latest versions
Keeps the latest version of each trade in silver.trades and moves older versions to quarantine.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy("trade_id").orderBy(F.col("updated_at").desc(), F.col("_ingested_at").desc())

ranked = spark.table("silver.trades").withColumn("rn", F.row_number().over(w))
ranked.cache()
print(ranked.count())   # keeps a copy in memory before we rewrite the table

In [0]:
latest = ranked.filter("rn = 1").drop("rn")
older  = ranked.filter("rn > 1").drop("rn")
display(older)

In [0]:
# older versions go to quarantine (same columns and text types as the quarantine table)
q_cols = spark.table("silver.trades_quarantine").columns
older_q = (older
    .withColumn("reject_reason", F.lit("DUPLICATE_TRADE_ID"))
    .withColumn("_rescued_data", F.lit(None))
    .select([F.col(c) if c in ("_ingested_at", "reject_reason") else F.col(c).cast("string") for c in q_cols]))
older_q.write.mode("append").saveAsTable("silver.trades_quarantine")

# Silver keeps only the latest version
latest.write.mode("overwrite").saveAsTable("silver.trades")

In [0]:
%sql
SELECT 'silver' AS tbl, COUNT(*) AS n FROM silver.trades
UNION ALL
SELECT 'quarantine', COUNT(*) FROM silver.trades_quarantine

In [0]:
%sql
SELECT trade_id, quantity, updated_at FROM silver.trades WHERE trade_id IN ('T005', 'T006')